In [1]:
## 📈 02: Exploratory Data Analysis (EDA) and Risk Factor Identification

# --- Setup ---
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np

# Set aesthetic parameters for plots
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# --- Configuration ---
PROCESSED_DATA_PATH = '../data/processed/patient_records_cleaned.csv'
VISUALS_DIR = '../docs/visuals'
os.makedirs(VISUALS_DIR, exist_ok=True)
df = pd.read_csv(PROCESSED_DATA_PATH)


In [ ]:
# --- 2. Univariate Distributions and Data Quality ---

print("\n--- Value Counts for Key Categorical Features ---")
print(df['LOS_Bucket'].value_counts())
print(df['Diagnosis_Category'].value_counts().head())

# TODO: Plot the distribution of Length of Stay (LOS)
plt.figure(figsize=(8, 5))
sns.histplot(df['Length_of_Stay'], bins=df['Length_of_Stay'].max(), kde=True)
plt.title('Distribution of Length of Stay (Days)')
plt.xlim(0, 14) # Focus on stays under 14 days for clarity
plt.savefig(os.path.join(VISUALS_DIR, 'los_distribution.png'))
plt.close()
BASE_RATE = df['Readmitted_30days'].mean()
print(f"Overall 30-Day Readmission Rate (Baseline): {BASE_RATE:.2%}")



--- Value Counts for Key Categorical Features ---
LOS_Bucket
Short (1-3d)     48147
Medium (4-7d)    36484
Long (8+d)       14712
Name: count, dtype: int64
Diagnosis_Category
Circulatory/Cardiovascular      29585
Other/Ill-defined               19218
Endocrine/Metabolic/Immunity    11304
Respiratory System               9919
Digestive System                 9072
Name: count, dtype: int64
Overall 30-Day Readmission Rate (Baseline): 11.39%


In [ ]:
# --- 3. Bivariate Analysis: Readmission Rate by Key Risk Factors ---

def plot_readmission_rate(data, col, title, file_name, base_rate, rotation=0):
    """Calculates and plots readmission rate by a given categorical column."""
    rate_df = data.groupby(col)['Readmitted_30days'].agg(['mean', 'count']).reset_index()
    rate_df = rate_df.rename(columns={'mean': 'Readmission_Rate', 'count': 'Total_Encounters'})
    rate_df = rate_df[rate_df['Total_Encounters'] > 500].sort_values('Readmission_Rate', ascending=False)

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(x=col, y='Readmission_Rate', hue=col, data=rate_df, palette="crest", legend=False)

    ax.axhline(base_rate, color='r', linestyle='--', label=f'Baseline ({base_rate:.2%})')

    plt.title(title, fontsize=14)
    plt.ylabel('30-Day Readmission Rate', fontsize=12)
    plt.xlabel(col, fontsize=12)
    plt.xticks(rotation=rotation, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(VISUALS_DIR, file_name))
    plt.close()



In [ ]:
# 3.1. Readmission by Diagnosis Category
plot_readmission_rate(
    df, 'Diagnosis_Category',
    'Readmission Rate by Primary Diagnosis Category (Top Risks)',
    'diagnosis_category_readmission.png',
    base_rate=BASE_RATE,
    rotation=45
)

In [27]:
# 3.2. Readmission by Age Group
# Ensure Age_Group is ordered correctly (categorical type helps here)
plot_readmission_rate(
    df, 'Age_Group', 
    'Readmission Rate by Patient Age Group',
    'age_group_readmission.png',
    base_rate=BASE_RATE,
    rotation=0
)


In [29]:
# 3.3. Readmission by Length of Stay (LOS) Bucket
# Note: This is an important visual for the Power BI trend line.
plot_readmission_rate(
    df, 'LOS_Bucket', 
    'Readmission Rate by Length of Stay Bucket',
    'los_bucket_readmission.png',
    base_rate=BASE_RATE,
    rotation=0
)


 4. Multivariate Analysis: Heatmaps and Cross-Tabulation ---

In [30]:
# 4.1. Age Group vs. Comorbidity Count Heatmap (Crucial Risk Visual)
# Create a pivot table where values are the mean readmission rate
pivot_data = df.pivot_table(
    index='Age_Group', 
    columns='Comorbidity_Count', 
    values='Readmitted_30days', 
    aggfunc='mean'
)

plt.figure(figsize=(10, 7))
sns.heatmap(
    pivot_data, 
    annot=True, 
    cmap='YlGnBu', 
    fmt=".2%", 
    linewidths=.5, 
    linecolor='white',
    cbar_kws={'label': '30-Day Readmission Rate'}
)
plt.title('Risk Heatmap: Readmission Rate by Age and Comorbidity Count', fontsize=14)
plt.ylabel('Age Group')
plt.xlabel('Comorbidity Count Proxy (from diag 2/3)')
plt.savefig(os.path.join(VISUALS_DIR, 'age_comorbidity_heatmap.png'))
plt.close()

In [31]:
# 4.2. Correlation Heatmap (for Numerical Features)
numerical_cols = [
    'Length_of_Stay', 'Num_Prior_Admissions', 'Num_Medications', 
    'Num_Lab_Procedures', 'Num_Procedures', 'Comorbidity_Count', 
    'Medication_Change', 'Readmitted_30days'
]
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".2f", 
    linewidths=.5,
    mask=np.triu(correlation_matrix) # Hide upper triangle for clarity
)
plt.title('Correlation Heatmap of Numerical Features and Readmission', fontsize=14)
plt.savefig(os.path.join(VISUALS_DIR, 'correlation_heatmap.png'))
plt.close()


print("\nEDA complete. Key visuals saved to the 'docs/visuals/' directory.")
# --- End of Notebook 02 ---


EDA complete. Key visuals saved to the 'docs/visuals/' directory.
